# CenterNet · Objects as Points

把物体建模为类别热图中的中心点。2D 检测头预测每类中心热图、中心偏移量和框的宽高。原文使用输出步长 4。训练时用高斯峰值表示中心，热图使用带惩罚项调整的 focal loss，偏移和尺寸使用在真实中心处计算的 L1 损失。

推理时在每张类别热图中用局部最大值筛选峰，再取 Top-K。示例展示单个峰的解码，假设宽高与中心偏移均以热图网格为单位。真实流水线还要把预测映射回输入预处理之前的坐标。

来源 [Objects as Points](https://arxiv.org/abs/1904.07850)。

In [ ]:
def decode_peak(column, row, offset_x, offset_y, width, height, stride=4):
    center_x, center_y = column + offset_x, row + offset_y
    return (
        (center_x - width / 2) * stride,
        (center_y - height / 2) * stride,
        (center_x + width / 2) * stride,
        (center_y + height / 2) * stride,
    )


assert decode_peak(10, 6, 0.25, 0.5, 8, 5) == (25.0, 16.0, 57.0, 36.0)